# PalakGPT – Mini GPT Language Model

Built using nanoGPT (Andrej Karpathy) as the base code for the transformer architecture.

Trained a small character-level model first on the Tiny Shakespeare dataset, then on my own multilingual dataset (Hindi + English + French + Japanese). Also tried out a custom sliding-window attention and a quick 127M param scale check. Added a simple web UI at the end to chat with the trained model.

In [1]:
# mounting drive since the repo and data are saved there
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# checking the nanoGPT folder is there
!ls /content/drive/MyDrive/

'Colab Notebooks'   palak_desai_resume.pdf
 nano-GPT	    PalakDesai_SanjivaniCollegeOfEngineeringKopargaon.mp4


In [2]:
# going into the actual repo folder
%cd /content/drive/MyDrive/nano-GPT/nanoGPT
!ls

/content/drive/MyDrive/nano-GPT/nanoGPT
assets		 model_local_attn.py   README.md
bench.py	 model.py	       sample.py
config		 out-hindi-char        scaling_laws.ipynb
configurator.py  out-hindi-local-attn  train_local_attn.py
data		 out-shakespeare-char  train.py
LICENSE		 __pycache__	       transformer_sizing.ipynb


## Installing requirements

In [4]:
!pip install -q torch numpy transformers datasets tiktoken tqdm flask pyngrok

## Part 1 — Shakespeare dataset (baseline)

nanoGPT already comes with a script to prepare the Tiny Shakespeare dataset, so using that to get a baseline character-level model first.

In [3]:
!python data/shakespeare_char/prepare.py

length of dataset in characters: 1,115,394
all the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size: 65
train has 1,003,854 tokens
val has 111,540 tokens


## Part 2 — Multilingual dataset (Hindi + English + French + Japanese)

Wanted to check if a character-level model can also learn from more than one language and script. Made a small text file mixing Hindi, English, French and Japanese sentences.

In [6]:
!mkdir -p data/hindi_char

In [7]:
%%writefile data/hindi_char/input.txt
नमस्ते दुनिया। यह एक बहुभाषी पाठ नमूना है जो चरित्र-स्तरीय भाषा मॉडल के प्रशिक्षण के लिए बनाया गया है। भारत एक विशाल देश है जहाँ अनेक भाषाएँ बोली जाती हैं। हिंदी, तमिल, बंगाली, मराठी और पंजाबी जैसी भाषाएँ मिलकर भारत की सांस्कृतिक विविधता को दर्शाती हैं। संगीत, नृत्य और त्योहार हर राज्य में अलग-अलग रूप में मनाए जाते हैं। दिवाली रोशनी का त्योहार है, और होली रंगों का त्योहार है। भारतीय भोजन में मसालों का विशेष महत्व है, जैसे हल्दी, जीरा, धनिया और मिर्च।

Hello world. This is a multilingual text sample created to train a character-level language model. It mixes Hindi, English, Japanese, and French to demonstrate that the tokenizer adapts to whatever characters appear in the file. Language models learn patterns from data, and the more varied and plentiful the data, the better the model can generalize instead of simply memorizing. Technology today connects people across the globe, allowing ideas to spread quickly between cultures. Food, music, and travel are common topics that people enjoy discussing no matter what language they speak. A good cup of coffee in the morning can make the whole day feel brighter. Traveling to new places teaches us about different customs and ways of life.

Bonjour le monde. Ceci est un exemple de texte multilingue. La France est connue pour sa cuisine, son art et son histoire riche. Paris, la capitale, attire des millions de visiteurs chaque année grâce à ses musées et ses monuments célèbres comme la tour Eiffel. Le café et le pain frais font partie intégrante de la culture française du matin. La musique et la littérature françaises ont influencé le monde entier pendant des siècles. Les voyages permettent de découvrir de nouvelles langues et de nouvelles façons de penser.

こんにちは世界。これは文字レベルの言語モデルを訓練するための多言語テキストサンプルです。日本は美しい四季を持つ国として知られています。春には桜が咲き、多くの人々が花見を楽しみます。夏は祭りと花火の季節です。秋には紅葉が山々を彩り、冬には雪が静かに降り積もります。日本の伝統文化には、茶道、書道、そして着物があります。都市部では現代的な技術と伝統が共存しており、東京はその代表的な例です。

ज्ञान ही शक्ति है। Knowledge is power. Le savoir, c'est le pouvoir. 知識は力なり。 सीखना एक निरंतर प्रक्रिया है जो जीवन भर चलती रहती है। Learning is a lifelong process that never truly ends. Apprendre est un processus qui dure toute la vie. 学ぶことは一生続くプロセスです。 हर दिन कुछ नया सीखने का प्रयास करना चाहिए। Every day is an opportunity to learn something new. Chaque jour est une occasion d'apprendre quelque chose de nouveau. 毎日は何か新しいことを学ぶ機会です。


Overwriting data/hindi_char/input.txt


In [8]:
# reusing the same prepare.py logic for the hindi dataset (builds vocab + train/val split)
!cp data/shakespeare_char/prepare.py data/hindi_char/prepare.py
!python data/hindi_char/prepare.py

length of dataset in characters: 2,352
all the unique characters: 
 ',-.ABCEFHIJKLPTabcdefghijklmnopqrstuvwxyzàâçèéँंअएऔकखगचछजञठडणतदधनपबभमयरलवशषसहािीुूृेैॉो्।、。々あいおかがきくこしすそたちつてでとなにのはぶまみめもらりるれをんキサスセテデトプベモルレロン一世京人代会伝何例共冬力化咲四国夏多字存季学山市彩技持文新日春書本東桜楽機毎火物現生界的着知祭秋積節紅統続練美花茶葉術表見言訓語識道部都降雪静
vocab size: 212
train has 2,116 tokens
val has 236 tokens


### Setting up the training config

In [9]:
!cp config/train_shakespeare_char.py config/train_hindi_char.py
!sed -i "s/dataset = 'shakespeare_char'/dataset = 'hindi_char'/" config/train_hindi_char.py
!sed -i "s/out_dir = 'out-shakespeare-char'/out_dir = 'out-hindi-char'/" config/train_hindi_char.py
!sed -i "s/max_iters = 5000/max_iters = 2000/" config/train_hindi_char.py
!sed -i "s/lr_decay_iters = 5000/lr_decay_iters = 2000/" config/train_hindi_char.py
!sed -i "s/block_size = 256/block_size = 64/" config/train_hindi_char.py
!sed -i "s/always_save_checkpoint = False/always_save_checkpoint = True/" config/train_hindi_char.py
!cat config/train_hindi_char.py

# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-hindi-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = True

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'hindi_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 64 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 2000
lr_decay_iters = 2000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of tokens per iter is small

warmup_iters = 100 # not super necessary pote

### Training the multilingual model

In [10]:
!python train.py config/train_hindi_char.py --compile=False

Overriding config with config/train_hindi_char.py:
# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-hindi-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = True

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'hindi_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 64 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 2000
lr_decay_iters = 2000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of tokens per iter is s

In [11]:
# just checking the checkpoint actually got saved
import torch
ckpt = torch.load('out-hindi-char/ckpt.pt', map_location='cpu')
print("saved at iter:", ckpt['iter_num'], "best val loss:", ckpt['best_val_loss'])

saved at iter: 2000 best val loss: tensor(8.0378)


In [12]:
!python sample.py --out_dir=out-hindi-char --num_samples=3 --max_new_tokens=200

Overriding: out_dir = out-hindi-char
Overriding: num_samples = 3
Overriding: max_new_tokens = 200
number of parameters: 10.70M
Loading meta from data/hindi_char/meta.pkl...


Hello world. This is a multilingual text sample created to train a character-level language model. It mixes Hindi, English, Japanese, and French to demonstrate that the tokenizer adapts to whatever c
---------------

Hello world. This is a multilingual text sample created to train a character-level language model. It mixes Hindi, English, Japanese, and French to demonstrate that the tokenizer adapts to whatever ch
---------------


Hello world. This is a multilingual text sample created to train a character-level language model. It mixes Hindi, English, Japanese, and French to demonstrate that the tokenizer adapts to whatever c
---------------


## Part 3 — Trying a sliding-window (local) attention

Normal GPT attention lets every token look back at all previous tokens. Wanted to try limiting that to only the last few tokens (sliding window / local attention) and compare the output with the normal full-attention model.

In [13]:
# making a copy so the original model.py stays untouched
!cp model.py model_local_attn.py
!grep -n "class CausalSelfAttention" -A 40 model_local_attn.py | head -60

29:class CausalSelfAttention(nn.Module):
30-
31-    def __init__(self, config):
32-        super().__init__()
33-        assert config.n_embd % config.n_head == 0
34-        # key, query, value projections for all heads, but in a batch
35-        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
36-        # output projection
37-        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
38-        # regularization
39-        self.attn_dropout = nn.Dropout(config.dropout)
40-        self.resid_dropout = nn.Dropout(config.dropout)
41-        self.n_head = config.n_head
42-        self.n_embd = config.n_embd
43-        self.dropout = config.dropout
44-        # flash attention make GPU go brrrrr but support is only in PyTorch >= 2.0
45-        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
46-        if not self.flash:
47-            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
48-

In [14]:
# adding a window_size option so attention only looks at the last N tokens
with open('model_local_attn.py') as f:
    src = f.read()

src = src.replace(
    "self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')",
    "self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')\n"
    "        self.window_size = getattr(config, 'window_size', None)  # None = normal full attention"
)

old_forward_block = '''        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # efficient attention using Flash Attention CUDA kernels
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:'''

new_forward_block = '''        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.window_size is not None:
            # sliding-window attention: causal mask + only look back window_size tokens
            idx = torch.arange(T, device=x.device)
            causal = idx.unsqueeze(0) <= idx.unsqueeze(1)
            local = idx.unsqueeze(0) > (idx.unsqueeze(1) - self.window_size)
            mask = (causal & local).view(1, 1, T, T)
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(~mask, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
        elif self.flash:
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:'''

assert old_forward_block in src, "pattern not found"
src = src.replace(old_forward_block, new_forward_block)

# also add window_size as a field on the config class
src = src.replace(
    "class GPTConfig:",
    "class GPTConfig:\n    window_size: int = None  # set this to use sliding-window attention"
)

with open('model_local_attn.py', 'w') as f:
    f.write(src)

print("model_local_attn.py updated")

model_local_attn.py updated


In [15]:
!cp config/train_hindi_char.py config/train_hindi_local_attn.py
!sed -i "s/out_dir = 'out-hindi-char'/out_dir = 'out-hindi-local-attn'/" config/train_hindi_local_attn.py
!echo "window_size = 8" >> config/train_hindi_local_attn.py
!cat config/train_hindi_local_attn.py

# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-hindi-local-attn'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = True

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'hindi_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 64 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 2000
lr_decay_iters = 2000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of tokens per iter is small

warmup_iters = 100 # not super necessar

In [16]:
# train.py doesn't pass window_size through by default, so wiring it into model_args
with open('train.py') as f:
    train_src = f.read()

old_line = "model_args = dict(n_layer=n_layer, n_head=n_head, n_embd=n_embd, block_size=block_size,\n                  bias=bias, vocab_size=None, dropout=dropout)"
new_line = "model_args = dict(n_layer=n_layer, n_head=n_head, n_embd=n_embd, block_size=block_size,\n                  bias=bias, vocab_size=None, dropout=dropout, window_size=globals().get('window_size', None))"

assert old_line in train_src, "line not found"
train_src = train_src.replace(old_line, new_line)

with open('train_local_attn.py', 'w') as f:
    f.write(train_src)

print("train_local_attn.py ready")

train_local_attn.py ready


In [17]:
%cd /content/drive/MyDrive/nano-GPT/nanoGPT
!ls model_local_attn.py train_local_attn.py config/train_hindi_local_attn.py

/content/drive/MyDrive/nano-GPT/nanoGPT
config/train_hindi_local_attn.py  model_local_attn.py  train_local_attn.py


In [18]:
import sys, importlib.util

spec = importlib.util.spec_from_file_location("model", "model_local_attn.py")
model_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_module)
sys.modules["model"] = model_module

sys.argv = ["train_local_attn.py", "config/train_hindi_local_attn.py", "--compile=False"]
exec(open("train_local_attn.py").read())

Overriding config with config/train_hindi_local_attn.py:
# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-hindi-local-attn'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = True

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'hindi_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 64 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 2000
lr_decay_iters = 2000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of tokens p

<string>:196: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.


using fused AdamW: True
step 0: train loss 5.3946, val loss 5.4233
iter 0: loss 5.4061, time 17017.82ms, mfu -100.00%
iter 10: loss 4.1094, time 119.48ms, mfu 0.73%
iter 20: loss 3.4326, time 119.30ms, mfu 0.73%
iter 30: loss 2.8958, time 119.76ms, mfu 0.72%
iter 40: loss 2.4809, time 120.40ms, mfu 0.72%
iter 50: loss 2.1373, time 121.91ms, mfu 0.72%
iter 60: loss 1.8022, time 123.13ms, mfu 0.72%
iter 70: loss 1.5217, time 120.62ms, mfu 0.72%
iter 80: loss 1.0940, time 121.35ms, mfu 0.72%
iter 90: loss 0.8840, time 121.29ms, mfu 0.72%
iter 100: loss 0.6189, time 122.33ms, mfu 0.72%
iter 110: loss 0.4643, time 121.20ms, mfu 0.72%
iter 120: loss 0.3583, time 121.34ms, mfu 0.72%
iter 130: loss 0.3119, time 122.73ms, mfu 0.72%
iter 140: loss 0.2730, time 120.45ms, mfu 0.72%
iter 150: loss 0.2594, time 120.71ms, mfu 0.72%
iter 160: loss 0.2300, time 120.68ms, mfu 0.72%
iter 170: loss 0.2239, time 120.43ms, mfu 0.72%
iter 180: loss 0.1938, time 120.45ms, mfu 0.72%
iter 190: loss 0.1976, time

In [19]:
# generate text from the local-attention checkpoint
sys.argv = ["sample.py", "--out_dir=out-hindi-local-attn", "--num_samples=3", "--max_new_tokens=200"]
exec(open("sample.py").read())

Overriding: out_dir = out-hindi-local-attn
Overriding: num_samples = 3
Overriding: max_new_tokens = 200
number of parameters: 10.70M
Loading meta from data/hindi_char/meta.pkl...


Hello world. This is a multilingual text sample created to train a character-level language model. It mixes Hindi, English, Japanese, and French to demonstrate that the tokenizer adapts to whatever c
---------------


Hello world. This is a multilingual text sample created to train a character-level language model. It mixes Hindi, English, Japanese, and French to demonstrate that the tokenizer adapts to whatever c
---------------


Hello world. This is a multilingual text sample created to train a character-level language model. It mixes Hindi, English, Japanese, and French to demonstrate that the tokenizer adapts to whatever c
---------------


### Checking the sliding-window mask math

In [20]:
import torch

spec = importlib.util.spec_from_file_location("model_local", "model_local_attn.py")
model_local = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_local)

T = 20
window_size = 8
idx = torch.arange(T)
causal = idx.unsqueeze(0) <= idx.unsqueeze(1)
local = idx.unsqueeze(0) > (idx.unsqueeze(1) - window_size)
mask = causal & local

# for token at position 15, which earlier positions can it see?
visible_positions = torch.nonzero(mask[15]).squeeze().tolist()
print(f"Token at position 15 can attend to positions: {visible_positions}")
print(f"Number of visible positions: {len(visible_positions)} (should be exactly {window_size})")

Token at position 15 can attend to positions: [8, 9, 10, 11, 12, 13, 14, 15]
Number of visible positions: 8 (should be exactly 8)


### Comparing full attention vs local attention output

In [21]:
print("=== FULL ATTENTION ===")
sys.argv = ["sample.py", "--out_dir=out-hindi-char", "--start=ज्ञान", "--num_samples=1", "--max_new_tokens=150"]
sys.modules["model"] = __import__("model")  # restore original model.py
exec(open("sample.py").read())

print("\n=== LOCAL ATTENTION (window=8) ===")
sys.modules["model"] = model_module  # swap back to the patched version
sys.argv = ["sample.py", "--out_dir=out-hindi-local-attn", "--start=ज्ञान", "--num_samples=1", "--max_new_tokens=150"]
exec(open("sample.py").read())

=== FULL ATTENTION ===
Overriding: out_dir = out-hindi-char
Overriding: start = ज्ञान
Overriding: num_samples = 1
Overriding: max_new_tokens = 150
number of parameters: 10.70M
Loading meta from data/hindi_char/meta.pkl...
ज्ञान ही शक्ति है। Knowledge is power. Le savoir, c'est le pouvoir. 知識は力なり。 सीखना एक निरंतर प्रक्रिया है जो जीवन भर चलती रहती है। Learning is a lifelong pr
---------------

=== LOCAL ATTENTION (window=8) ===
Overriding: out_dir = out-hindi-local-attn
Overriding: start = ज्ञान
Overriding: num_samples = 1
Overriding: max_new_tokens = 150
number of parameters: 10.70M
Loading meta from data/hindi_char/meta.pkl...
ज्ञान ही शक्ति है। Knowledge is power. Le savoir, c'est le pouvoir. 知識は力なり。 सीखना एक निरंतर प्रक्रिया है जो जीवन भर चलती रहती है। Learning is a lifelong pr
---------------


## Part 4 — Quick check: does it scale to full GPT-2 size?

Not actually training a 127M parameter model here (way too much compute for a Colab GPU), just confirming the same code can build a full-size GPT-2 config and run a few training steps without crashing.

In [22]:
import torch
from model import GPTConfig, GPT

gpt2_config = GPTConfig(
    block_size=1024,
    vocab_size=50304,
    n_layer=12,
    n_head=12,
    n_embd=768,
    dropout=0.1,
    bias=True,
)
model = GPT(gpt2_config)
n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params/1e6:.2f}M")

number of parameters: 123.69M
Total parameters: 124.48M


In [23]:
import time
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

x = torch.randint(0, gpt2_config.vocab_size, (4, 64), device=device)
y = torch.randint(0, gpt2_config.vocab_size, (4, 64), device=device)

for step in range(10):
    t0 = time.time()
    logits, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"step {step}: loss {loss.item():.4f}, time {(time.time()-t0)*1000:.1f}ms")

step 0: loss 10.9680, time 183.9ms
step 1: loss 9.7516, time 105.5ms
step 2: loss 9.2097, time 98.7ms
step 3: loss 8.8545, time 98.2ms
step 4: loss 8.7503, time 99.7ms
step 5: loss 8.3108, time 98.5ms
step 6: loss 7.8091, time 99.6ms
step 7: loss 7.5954, time 97.4ms
step 8: loss 7.0245, time 97.5ms
step 9: loss 6.1349, time 98.7ms


## Web UI to chat with the model

Wrote a small local server plus an HTML/JS frontend so I can type a prompt in the browser and get generated text back, instead of running sample.py from the terminal every time.

In [6]:
# clearing any old process that might already be using this port
!fuser -k 3001/tcp 2>/dev/null || true
import time
time.sleep(2)

In [7]:
%%writefile /content/drive/MyDrive/nano-GPT/frontend/server.py
import os
import pickle
import json
from contextlib import nullcontext
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler
from urllib.parse import urlparse, parse_qs

import torch

FRONTEND_ROOT = os.path.dirname(os.path.abspath(__file__))
PALAKGPT_ROOT = os.path.dirname(FRONTEND_ROOT)
NANOGPT_DIR = os.path.join(PALAKGPT_ROOT, "nanoGPT")

import sys
sys.path.insert(0, NANOGPT_DIR)
from model import GPTConfig, GPT  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "bfloat16" if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else "float16" if DEVICE == "cuda" else "float32"
SEED = 1337

MAX_NEW_TOKENS = 300

torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed(SEED)

device_type = "cuda" if "cuda" in DEVICE else "cpu"
ptdtype = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[DTYPE]
ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

MODEL_REGISTRY = {
    "shakespeare": {
        "label": "Shakespeare (English)",
        "out_dir": os.path.join(NANOGPT_DIR, "out-shakespeare-char"),
    },
    "hindi": {
        "label": "Multilingual (Hindi/JP/FR demo)",
        "out_dir": os.path.join(NANOGPT_DIR, "out-hindi-char"),
    },
}


def load_model(ckpt_path):
    print(f"Loading checkpoint from {ckpt_path} ...")
    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    gptconf = GPTConfig(**checkpoint["model_args"])
    model = GPT(gptconf)
    state_dict = checkpoint["model"]
    unwanted_prefix = "_orig_mod."
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    model.load_state_dict(state_dict)
    model.eval()
    model.to(DEVICE)
    print("Model loaded.")
    return model, checkpoint


def load_tokenizer(checkpoint):
    meta_path = None
    if "config" in checkpoint and "dataset" in checkpoint["config"]:
        candidate = os.path.join(NANOGPT_DIR, "data", checkpoint["config"]["dataset"], "meta.pkl")
        if os.path.exists(candidate):
            meta_path = candidate

    if meta_path:
        print(f"Using char-level tokenizer from {meta_path}")
        with open(meta_path, "rb") as f:
            meta = pickle.load(f)
        stoi, itos = meta["stoi"], meta["itos"]

        def encode(s):
            return [stoi[c] for c in s if c in stoi]

        def decode(tokens):
            return "".join(itos[t] for t in tokens)

        return encode, decode

    print("meta.pkl not found — falling back to GPT-2 BPE tokenizer (tiktoken)")
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    encode = lambda s: enc.encode(s, allowed_special={"<|endoftext|>"})
    decode = lambda tokens: enc.decode(tokens)
    return encode, decode


LOADED = {}
for key, info in MODEL_REGISTRY.items():
    ckpt_path = os.path.join(info["out_dir"], "ckpt.pt")
    if not os.path.exists(ckpt_path):
        print(f"Skipping '{key}': no checkpoint at {ckpt_path}")
        continue
    model, checkpoint = load_model(ckpt_path)
    encode, decode = load_tokenizer(checkpoint)
    LOADED[key] = {"model": model, "checkpoint": checkpoint, "encode": encode, "decode": decode}

DEFAULT_MODEL_KEY = next(iter(LOADED)) if LOADED else None


def generate_reply(prompt: str, mode: str = "creative", model_key: str = None) -> str:
    if not prompt.strip():
        return "Please enter a prompt."

    model_key = model_key if model_key in LOADED else DEFAULT_MODEL_KEY
    if model_key is None:
        return "No model loaded on the server."

    entry = LOADED[model_key]
    model_obj, encode_fn, decode_fn = entry["model"], entry["encode"], entry["decode"]

    mode_params = {
        "creative": dict(temperature=1.0, top_k=200),
        "concise": dict(temperature=0.5, top_k=50),
        "technical": dict(temperature=0.7, top_k=100),
    }
    params = mode_params.get(mode, mode_params["creative"])

    ids = encode_fn(prompt)
    if not ids:
        return "Couldn't encode that prompt with the model's tokenizer — try different text."

    x = torch.tensor(ids, dtype=torch.long, device=DEVICE)[None, ...]

    with torch.no_grad():
        with ctx:
            y = model_obj.generate(
                x,
                MAX_NEW_TOKENS,
                temperature=params["temperature"],
                top_k=params["top_k"],
            )

    full_text = decode_fn(y[0].tolist())
    completion = full_text[len(prompt):] if full_text.startswith(prompt) else full_text
    return completion.strip() or full_text.strip()


class Handler(SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory=FRONTEND_ROOT, **kwargs)

    def do_GET(self):
        if self.path.startswith("/api/info"):
            qs = parse_qs(urlparse(self.path).query)
            requested = qs.get("model", [DEFAULT_MODEL_KEY])[0]
            model_key = requested if requested in LOADED else DEFAULT_MODEL_KEY

            entry = LOADED.get(model_key)
            if entry is None:
                info = {"models": [], "default": None, "current": None}
            else:
                model_args = entry["checkpoint"].get("model_args", {})
                dataset_name = entry["checkpoint"].get("config", {}).get("dataset", model_key)
                n_params = sum(p.numel() for p in entry["model"].parameters())
                info = {
                    "models": [{"key": k, "label": MODEL_REGISTRY[k]["label"]} for k in LOADED],
                    "default": DEFAULT_MODEL_KEY,
                    "current": {
                        "key": model_key,
                        "architecture": f"{model_args.get('n_layer', '?')} layers . "
                                         f"{model_args.get('n_head', '?')} heads . "
                                         f"{model_args.get('n_embd', '?')} dim",
                        "params": f"{n_params / 1e6:.2f}M",
                        "tokenizer": "character-level",
                        "dataset": dataset_name.replace("_", " ").title(),
                    },
                }
            body = json.dumps(info).encode("utf-8")
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(body)
            return
        super().do_GET()

    def do_POST(self):
        if self.path == "/api/generate":
            length = int(self.headers.get("Content-Length", 0))
            body = self.rfile.read(length).decode("utf-8")
            payload = json.loads(body or "{}")
            prompt = payload.get("prompt", "")
            mode = payload.get("mode", "creative")
            model_key = payload.get("model")

            try:
                reply_text = generate_reply(prompt, mode, model_key)
                status = 200
            except Exception as e:
                reply_text = f"Generation error: {e}"
                status = 500

            reply = {"reply": reply_text}
            self.send_response(status)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps(reply).encode("utf-8"))
            return
        self.send_error(404)


if __name__ == "__main__":
    port = int(os.environ.get("PORT", 3000))
    ThreadingHTTPServer.allow_reuse_address = True
    print(f"Serving frontend at http://localhost:{port}")
    ThreadingHTTPServer(("0.0.0.0", port), Handler).serve_forever()

Overwriting /content/drive/MyDrive/nano-GPT/frontend/server.py


In [8]:
%%writefile /content/drive/MyDrive/nano-GPT/frontend/index.html
<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>PalakGPT - a transformer built from scratch</title>
    <link rel="preconnect" href="https://fonts.googleapis.com" />
    <link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,500;9..144,600&family=JetBrains+Mono:wght@400;500;600&display=swap" rel="stylesheet" />
    <link rel="stylesheet" href="styles.css" />
  </head>
  <body>
    <div class="app-shell">
      <aside class="sidebar">
        <div class="brand">
          <div class="brand-mark">P</div>
          <div>
            <h1>PalakGPT</h1>
            <p>a decoder-only transformer, trained from scratch</p>
          </div>
        </div>
        <div class="panel">
          <h2>Try a prompt</h2>
          <div class="chip-list" id="examplePrompts"></div>
        </div>
        <div class="panel stats-panel">
          <h2>Model</h2>
          <div class="stat-card">
            <strong>Architecture</strong>
            <span id="statArch">-</span>
          </div>
          <div class="stat-card">
            <strong>Parameters</strong>
            <span id="statParams">-</span>
          </div>
          <div class="stat-card">
            <strong>Tokenizer</strong>
            <span id="statTokenizer">-</span>
          </div>
          <div class="stat-card">
            <strong>Trained on</strong>
            <span id="statData">-</span>
          </div>
        </div>
      </aside>
      <main class="main-panel">
        <header class="topbar">
          <div>
            <p class="eyebrow">Playground</p>
            <h2>Give it an opening line</h2>
          </div>
          <button id="clearBtn" class="secondary-btn">Clear chat</button>
        </header>
        <section class="chat-window" id="chatWindow" aria-live="polite"></section>
        <section class="composer">
          <label class="visually-hidden" for="promptInput">Enter a prompt</label>
          <textarea id="promptInput" rows="4" placeholder="ROMEO:"></textarea>
          <div class="composer-actions">
            <select id="modelSelect" aria-label="Select model"></select>
            <select id="modeSelect" aria-label="Select response mode">
              <option value="creative">Creative</option>
              <option value="concise">Concise</option>
              <option value="technical">Technical</option>
            </select>
            <button id="generateBtn">Generate</button>
          </div>
        </section>
      </main>
    </div>
    <script src="app.js"></script>
  </body>
</html>

Overwriting /content/drive/MyDrive/nano-GPT/frontend/index.html


In [9]:
%%writefile /content/drive/MyDrive/nano-GPT/frontend/app.js
const examples = [
  "ROMEO:",
  "To be, or not to be,",
  "First Citizen:"
];
const chatWindow = document.getElementById('chatWindow');
const promptInput = document.getElementById('promptInput');
const generateBtn = document.getElementById('generateBtn');
const clearBtn = document.getElementById('clearBtn');
const examplePrompts = document.getElementById('examplePrompts');
const modeSelect = document.getElementById('modeSelect');
const modelSelect = document.getElementById('modelSelect');

function renderExamples() {
  examplePrompts.innerHTML = examples
    .map((example) => `<button class="chip" data-example="${example}">${example}</button>`)
    .join('');
  examplePrompts.querySelectorAll('.chip').forEach((button) => {
    button.addEventListener('click', () => {
      promptInput.value = button.dataset.example;
      promptInput.focus();
    });
  });
}

function appendMessage(text, role = 'assistant') {
  const message = document.createElement('div');
  message.className = `message ${role}`;
  message.textContent = text;
  chatWindow.appendChild(message);
  chatWindow.scrollTop = chatWindow.scrollHeight;
}

function setLoadingState(isLoading) {
  generateBtn.disabled = isLoading;
  generateBtn.textContent = isLoading ? 'Generating...' : 'Generate';
}

function createOfflineNotice() {
  return '[offline demo mode - backend unavailable. Start server.py and reload.]';
}

async function generateReply(prompt, mode, modelKey) {
  try {
    const response = await fetch('/api/generate', {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify({ prompt, mode, model: modelKey }),
    });
    if (!response.ok) throw new Error('Backend unavailable');
    const data = await response.json();
    return data.reply || data.message || 'No reply returned.';
  } catch (error) {
    return createOfflineNotice();
  }
}

async function handleGenerate() {
  const prompt = promptInput.value.trim();
  if (!prompt) {
    promptInput.focus();
    return;
  }
  appendMessage(prompt, 'user');
  promptInput.value = '';
  setLoadingState(true);
  const mode = modeSelect.value;
  const modelKey = modelSelect.value;
  const reply = await generateReply(prompt, mode, modelKey);
  appendMessage(reply, 'assistant');
  setLoadingState(false);
}

function clearChat() {
  chatWindow.innerHTML = '';
  appendMessage('Give me an opening line and I\u2019ll continue it in the style I learned.', 'assistant');
}

async function loadModelInfo(modelKey) {
  try {
    const url = modelKey ? `/api/info?model=${encodeURIComponent(modelKey)}` : '/api/info';
    const res = await fetch(url);
    if (!res.ok) return;
    const info = await res.json();

    if (modelSelect.options.length === 0 && Array.isArray(info.models)) {
      modelSelect.innerHTML = info.models
        .map((m) => `<option value="${m.key}">${m.label}</option>`)
        .join('');
      modelSelect.value = info.default;
    }

    const c = info.current;
    if (c) {
      document.getElementById('statArch').textContent = c.architecture ?? '—';
      document.getElementById('statParams').textContent = c.params ?? '—';
      document.getElementById('statTokenizer').textContent = c.tokenizer ?? '—';
      document.getElementById('statData').textContent = c.dataset ?? '—';
    }
  } catch (e) {
    // backend not running yet - ignore
  }
}

generateBtn.addEventListener('click', handleGenerate);
clearBtn.addEventListener('click', clearChat);
modelSelect.addEventListener('change', () => loadModelInfo(modelSelect.value));
promptInput.addEventListener('keydown', (event) => {
  if (event.key === 'Enter' && !event.shiftKey) {
    event.preventDefault();
    handleGenerate();
  }
});
renderExamples();
clearChat();
loadModelInfo();

Overwriting /content/drive/MyDrive/nano-GPT/frontend/app.js


In [17]:
# starting the server in the background
import subprocess, os, time
proc = subprocess.Popen(
    ["python", "/content/drive/MyDrive/nano-GPT/frontend/server.py"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    preexec_fn=os.setpgrp,
    env={**os.environ, "PORT": "3001"},
)
time.sleep(10)
print("still running:", proc.poll())

still running: None


In [18]:
# quick check that the server is actually responding
!curl -s http://localhost:3001/api/info

{"models": [{"key": "shakespeare", "label": "Shakespeare (English)"}, {"key": "hindi", "label": "Multilingual (Hindi/JP/FR demo)"}], "default": "shakespeare", "current": {"key": "shakespeare", "architecture": "6 layers . 6 heads . 384 dim", "params": "10.75M", "tokenizer": "character-level", "dataset": "Shakespeare Char"}}

In [19]:
# grabbing the public colab url to open in the browser
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(3001)"))

https://3001-gpu-t4-s-kkb-ass1c2-1oxbjh0k55fky-c.asia-southeast1-2.prod.colab.dev


## Stopping the server

In [ ]:
!fuser -k 3001/tcp 2>/dev/null || true
import time
time.sleep(2)
!lsof -i :3001

In [15]:
!kill -9 8566

In [16]:
!lsof -i :3001